# Day 56 — Advanced RAG
### Citation tracking, query transformation, HyDE, multi-turn conversation

## 1. Setup — Reuse Day 55 Infrastructure

In [1]:
import chromadb
import anthropic
from sentence_transformers import SentenceTransformer
import json

client = anthropic.Anthropic()
MODEL = "claude-haiku-4-5"
embedder = SentenceTransformer("all-MiniLM-L6-v2")

# rebuild the same corpus from Day 55
documents = [
    {"id": "doc1", "title": "Python Programming",
     "text": """Python is a high-level, interpreted programming language known for its simplicity and readability.
It was created by Guido van Rossum and first released in 1991. Python supports multiple programming
paradigms including procedural, object-oriented, and functional programming. It has a large standard
library and a vibrant ecosystem of third-party packages. Python is widely used in web development,
data science, artificial intelligence, automation, and scientific computing."""},
    {"id": "doc2", "title": "Machine Learning",
     "text": """Machine learning is a subset of artificial intelligence that enables systems to learn and improve
from experience without being explicitly programmed. The main types are supervised learning, unsupervised
learning, and reinforcement learning. Common algorithms include linear regression, decision trees,
random forests, support vector machines, and neural networks. Machine learning is used in spam
detection, image recognition, recommendation systems, and natural language processing."""},
    {"id": "doc3", "title": "Large Language Models",
     "text": """Large language models (LLMs) are deep learning models trained on massive text datasets to understand
and generate human language. They use transformer architecture with attention mechanisms to process
text. Examples include GPT-4, Claude, and Gemini. LLMs are trained using self-supervised learning
on billions of tokens of text. They can perform tasks like text generation, translation, summarisation,
question answering, and code generation. Fine-tuning adapts pre-trained LLMs for specific tasks."""},
    {"id": "doc4", "title": "Vector Databases",
     "text": """Vector databases are specialised databases designed to store and query high-dimensional vector
embeddings efficiently. Unlike traditional databases that use exact matching, vector databases use
approximate nearest neighbour search to find semantically similar items. Popular vector databases
include Pinecone, Weaviate, Qdrant, and ChromaDB. They are essential for semantic search, RAG
systems, and recommendation engines. Vectors are indexed using algorithms like HNSW for fast
approximate search. ChromaDB is an open-source, embeddable vector database for local development."""}
]

def fixed_size_chunk(text, chunk_size=40, overlap=5):
    words = text.split()
    chunks = []
    start = 0
    while start < len(words):
        end = min(start + chunk_size, len(words))
        chunks.append(" ".join(words[start:end]))
        start += chunk_size - overlap
    return chunks

chroma_client = chromadb.Client()
collection = chroma_client.create_collection(name="tech_docs")

all_chunks, all_ids, all_metadata = [], [], []
for doc in documents:
    chunks = fixed_size_chunk(doc["text"])
    for i, chunk in enumerate(chunks):
        all_chunks.append(chunk)
        all_ids.append(f"{doc['id']}_chunk{i}")
        all_metadata.append({"source": doc["id"], "title": doc["title"], "chunk_index": i})

embeddings = embedder.encode(all_chunks).tolist()
collection.add(documents=all_chunks, embeddings=embeddings, ids=all_ids, metadatas=all_metadata)

print(f"Ready — {collection.count()} chunks indexed")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Ready — 9 chunks indexed


## 2. Feature 1 — Citation Tracking

In [2]:
def retrieve(query, n_results=3):
    query_embedding = embedder.encode([query]).tolist()
    results = collection.query(
        query_embeddings=query_embedding,
        n_results=n_results,
        include=["documents", "metadatas", "distances"]
    )
    retrieved = []
    for i in range(len(results["ids"][0])):
        retrieved.append({
            "id": results["ids"][0][i],
            "text": results["documents"][0][i],
            "title": results["metadatas"][0][i]["title"],
            "chunk_index": results["metadatas"][0][i]["chunk_index"],
            "distance": round(results["distances"][0][i], 4)
        })
    return retrieved

def rag_with_citations(question, n_results=3):
    chunks = retrieve(question, n_results)

    numbered_context = ""
    for i, chunk in enumerate(chunks):
        numbered_context += f"[{i+1}] ({chunk['title']}): {chunk['text']}\n\n"

    prompt = f"""Answer the question using ONLY the context below.
After each claim in your answer, add a citation like [1] or [2] referring to the source number.
If the answer is not in the context, say 'I don't have enough information.'

Context:
{numbered_context}
Question: {question}"""

    response = client.messages.create(
        model=MODEL, max_tokens=400,
        messages=[{"role": "user", "content": prompt}]
    )

    answer = response.content[0].text
    citations = [{"ref": i+1, "title": c["title"], "chunk": c["id"]} for i, c in enumerate(chunks)]
    return {"answer": answer, "citations": citations}

result = rag_with_citations("What programming paradigms does Python support?")
print("ANSWER:\n", result["answer"])
print("\nCITATIONS:")
for c in result["citations"]:
    print(f"  [{c['ref']}] {c['title']} — {c['chunk']}")

ANSWER:
 According to the context, Python supports multiple programming paradigms including procedural, object-oriented, and functional programming. [1]

CITATIONS:
  [1] Python Programming — doc1_chunk0
  [2] Python Programming — doc1_chunk1
  [3] Machine Learning — doc2_chunk0


## 3. Feature 2 — Query Transformation

In [3]:
def transform_query(original_query):
    prompt = f"""You are a search query optimizer for a technical knowledge base.
Rewrite the following question to improve document retrieval.
Make it more specific, use technical terminology, and expand abbreviations.
Return ONLY the rewritten query, nothing else.

Original query: {original_query}"""

    response = client.messages.create(
        model=MODEL, max_tokens=100,
        messages=[{"role": "user", "content": prompt}]
    )
    return response.content[0].text.strip()

queries = [
    "How do LLMs work?",
    "What's the diff between supervised and unsupervised?",
    "Tell me about vector DBs"
]

print("Query Transformation Results:\n")
for q in queries:
    transformed = transform_query(q)
    print(f"Original:    {q}")
    print(f"Transformed: {transformed}")
    print()

Query Transformation Results:

Original:    How do LLMs work?
Transformed: How do Large Language Models work in terms of transformer architecture, neural network training, attention mechanisms, and token prediction?

Original:    What's the diff between supervised and unsupervised?
Transformed: What is the difference between supervised learning and unsupervised learning algorithms?

Original:    Tell me about vector DBs
Transformed: What are vector databases and how do they store and retrieve high-dimensional vector embeddings for semantic search and similarity matching?



## 4. Feature 3 — HyDE (Hypothetical Document Embeddings)

In [4]:
def hyde_retrieve(question, n_results=3):
    # Step 1: generate a hypothetical answer
    hyde_prompt = f"""Write a short, factual paragraph that would directly answer this question.
Write it as if it were an excerpt from a technical document.
Question: {question}"""

    hyp_response = client.messages.create(
        model=MODEL, max_tokens=150,
        messages=[{"role": "user", "content": hyde_prompt}]
    )
    hypothetical_answer = hyp_response.content[0].text.strip()
    print(f"Hypothetical answer:\n{hypothetical_answer}\n")

    # Step 2: embed the hypothetical answer (not the question)
    hyp_embedding = embedder.encode([hypothetical_answer]).tolist()
    results = collection.query(
        query_embeddings=hyp_embedding,
        n_results=n_results,
        include=["documents", "metadatas", "distances"]
    )

    retrieved = []
    for i in range(len(results["ids"][0])):
        retrieved.append({
            "title": results["metadatas"][0][i]["title"],
            "text": results["documents"][0][i],
            "distance": round(results["distances"][0][i], 4)
        })
    return retrieved

print("=== Standard retrieval ===")
standard = retrieve("What algorithms are used in machine learning?")
for r in standard:
    print(f"  [{r['distance']}] {r['title']}: {r['text'][:80]}...")

print("\n=== HyDE retrieval ===")
hyde = hyde_retrieve("What algorithms are used in machine learning?")
for r in hyde:
    print(f"  [{r['distance']}] {r['title']}: {r['text'][:80]}...")

=== Standard retrieval ===
  [0.461] Machine Learning: Machine learning is a subset of artificial intelligence that enables systems to ...
  [0.6208] Machine Learning: linear regression, decision trees, random forests, support vector machines, and ...
  [1.2729] Vector Databases: databases include Pinecone, Weaviate, Qdrant, and ChromaDB. They are essential f...

=== HyDE retrieval ===
Hypothetical answer:
# Machine Learning Algorithms

Machine learning employs a diverse set of algorithms broadly categorized into three primary types: supervised learning, unsupervised learning, and reinforcement learning. Supervised learning algorithms include linear regression, logistic regression, decision trees, random forests, support vector machines (SVMs), and neural networks, which train on labeled datasets to predict outputs. Unsupervised learning algorithms such as k-means clustering, hierarchical clustering, principal component analysis (PCA), and autoencoders discover patterns in unlabeled da

## 5. Feature 4 — Multi-Turn Conversation

In [5]:
def rag_chat(conversation_history, new_question, n_results=3):
    # transform query for better retrieval
    transformed = transform_query(new_question)
    chunks = retrieve(transformed, n_results)

    context = "\n\n".join([f"[{c['title']}]: {c['text']}" for c in chunks])

    system = f"""You are a helpful technical assistant. Answer questions using ONLY the context below.
If the answer is not in the context, say so clearly.

Context:
{context}"""

    # add new question to history
    conversation_history.append({"role": "user", "content": new_question})

    response = client.messages.create(
        model=MODEL, max_tokens=300,
        system=system,
        messages=conversation_history
    )

    answer = response.content[0].text
    conversation_history.append({"role": "assistant", "content": answer})
    return answer, conversation_history

# start a fresh conversation
history = []

q1 = "What is machine learning?"
answer1, history = rag_chat(history, q1)
print(f"Q1: {q1}")
print(f"A1: {answer1}\n")

q2 = "What are the main types you just mentioned?"
answer2, history = rag_chat(history, q2)
print(f"Q2: {q2}")
print(f"A2: {answer2}\n")

q3 = "Which of those is used in spam detection?"
answer3, history = rag_chat(history, q3)
print(f"Q3: {q3}")
print(f"A3: {answer3}")

Q1: What is machine learning?
A1: Based on the context provided:

Machine learning is a subset of artificial intelligence that enables systems to learn and improve from experience without being explicitly programmed.

The main types are:
- Supervised learning
- Unsupervised learning
- Reinforcement learning

Common algorithms include:
- Linear regression
- Decision trees
- Random forests
- Support vector machines
- Neural networks

Machine learning is applied in various domains such as spam detection, image recognition, recommendation systems, and natural language processing.

Q2: What are the main types you just mentioned?
A2: Based on the context provided, the main types of machine learning are:

1. **Supervised learning**
2. **Unsupervised learning**
3. **Reinforcement learning**

However, the context does not provide detailed explanations of what each of these types involves. I can only tell you their names from the information given.

Q3: Which of those is used in spam detection?


## 6. Summary — What We Built Today

| Feature | What it does | Why it matters |
|---------|-------------|----------------|
| Citation tracking | Numbers each source, model adds [1][2] references in answer | Users can verify every claim — essential for trust in production |
| Query transformation | Rewrites vague queries into precise technical questions | "Tell me about vector DBs" retrieves far more relevant chunks |
| HyDE | Generates hypothetical answer, searches with that embedding | Richer vocabulary in hypothetical answer = better chunk matches |
| Multi-turn conversation | Full history sent on every call | Follow-up questions like "the types you mentioned" resolve correctly |

**Key insight:**
Each technique addresses a different failure mode of basic RAG:
- Basic RAG fails on vague queries → query transformation fixes it
- Basic RAG fails on abstract questions → HyDE fixes it
- Basic RAG has no memory → conversation history fixes it
- Basic RAG has no accountability → citation tracking fixes it